In [1]:
!pip install pandas scikit-learn openai

In [2]:
import pandas as pd

data = {
    "login_days": [20, 3, 15, 2, 25, 7, 18, 4, 22, 5],
    "session_minutes": [45, 10, 35, 8, 60, 15, 40, 12, 55, 9],
    "features_used": [8, 2, 6, 1, 9, 3, 7, 2, 8, 2],
    "notifications_opened": [15, 1, 10, 0, 20, 3, 12, 1, 18, 2],
    "retained": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
}

df = pd.DataFrame(data)

df

,login_days,session_minutes,features_used,notifications_opened,retained
0,20,45,8,15,1
1,3,10,2,1,0
2,15,35,6,10,1
3,2,8,1,0,0
4,25,60,9,20,1
5,7,15,3,3,0
6,18,40,7,12,1
7,4,12,2,1,0
8,22,55,8,18,1
9,5,9,2,2,0


In [3]:
X = df[
    [
        "login_days",
        "session_minutes",
        "features_used",
        "notifications_opened"
    ]
]

y = df["retained"]

print("Input Features:")
print(X)

print("\nTarget:")
print(y)

Input Features:
   login_days  session_minutes  features_used  notifications_opened
0          20               45              8                    15
1           3               10              2                     1
2          15               35              6                    10
3           2                8              1                     0
4          25               60              9                    20
5           7               15              3                     3
6          18               40              7                    12
7           4               12              2                     1
8          22               55              8                    18
9           5                9              2                     2

Target:
0    1
1    0
2    1
3    0
4    1
5    0
6    1
7    0
8    1
9    0
Name: retained, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

print("Training data:", len(X_train))
print("Testing data:", len(X_test))


Training data: 7
Testing data: 3


In [5]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [6]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy * 100, 2), "%")

Model Accuracy: 100.0 %


In [7]:
new_user = [[
    12,   # login days
    30,   # session minutes
    5,    # features used
    8     # notifications opened
]]

prediction = model.predict(new_user)

if prediction[0] == 1:
    print("Prediction: User is likely to be retained")
else:
    print("Prediction: User may be at risk of churn")

Prediction: User is likely to be retained


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [8]:
new_user = pd.DataFrame([{
    "login_days": 12,
    "session_minutes": 30,
    "features_used": 5,
    "notifications_opened": 8
}])

prediction = model.predict(new_user)

if prediction[0] == 1:
    print("Prediction: User is likely to be retained")
else:
    print("Prediction: User may be at risk of churn")

Prediction: User is likely to be retained


In [9]:
journal_entries = [
    "Today I had a great day at college and enjoyed spending time with my friends.",
    "I was a little tired today, but overall my day was okay.",
    "Today was difficult and I felt frustrated throughout the day.",
    "Nothing seemed enjoyable today and I felt very negative."
]

for i, entry in enumerate(journal_entries, 1):
    print(f"Day {i}: {entry}")

Day 1: Today I had a great day at college and enjoyed spending time with my friends.
Day 2: I was a little tired today, but overall my day was okay.
Day 3: Today was difficult and I felt frustrated throughout the day.
Day 4: Nothing seemed enjoyable today and I felt very negative.


In [10]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print("API key loaded successfully!")
else:
    print("API key not found.")

API key loaded successfully!


In [11]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client connected successfully!")

OpenAI client connected successfully!


In [12]:
journal = """
Today was difficult and I felt frustrated throughout the day.
Nothing seemed enjoyable, and I felt very negative.
"""

response = client.responses.create(
    model="gpt-5",
    input=f"""
Analyze the emotional tone of this journal entry.

Journal:
{journal}

Give a simple response with:
1. Sentiment: Positive, Neutral, or Negative
2. Emotional trend
3. Whether a negative shift is present
4. Confidence: Low, Medium, or High

Do not diagnose any mental-health condition.
Do not make assumptions about the person.
"""
)

print(response.output_text)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [13]:
journal = """
Today was difficult and I felt frustrated throughout the day.
Nothing seemed enjoyable, and I felt very negative.
"""

print("Journal Entry:")
print(journal)

Journal Entry:

Today was difficult and I felt frustrated throughout the day.
Nothing seemed enjoyable, and I felt very negative.



In [14]:
from textblob import TextBlob

sentiment_score = TextBlob(journal).sentiment.polarity

if sentiment_score > 0.1:
    sentiment = "Positive"
elif sentiment_score < -0.1:
    sentiment = "Negative"
else:
    sentiment = "Neutral"

print("Sentiment:", sentiment)
print("Sentiment Score:", round(sentiment_score, 2))

Sentiment: Negative
Sentiment Score: -0.27


In [15]:
previous_entries = """
I had a great day at college and enjoyed spending time with my friends.
I was a little tired today, but overall my day was okay.
"""

previous_score = TextBlob(previous_entries).sentiment.polarity

print("Previous Sentiment Score:", round(previous_score, 2))
print("Current Sentiment Score:", round(sentiment_score, 2))

if sentiment_score < previous_score - 0.2:
    print("Negative shift detected")
else:
    print("No significant negative shift detected")

Previous Sentiment Score: 0.2
Current Sentiment Score: -0.27
Negative shift detected


In [16]:
safety_words = [
    "suicide",
    "kill myself",
    "end my life",
    "self harm",
    "hurt myself"
]

journal_lower = journal.lower()

safety_flag = any(word in journal_lower for word in safety_words)

if safety_flag:
    print("Safety concern detected.")
    print("Please reach out to a trusted adult or appropriate emergency/crisis support.")
else:
    print("No immediate safety concern detected.")

No immediate safety concern detected.


In [17]:
print("===== MOOD JOURNAL ANALYSIS =====")

print("Journal:")
print(journal)

print("\nCurrent Sentiment:", sentiment)
print("Current Score:", round(sentiment_score, 2))
print("Previous Score:", round(previous_score, 2))

if sentiment_score < previous_score - 0.2:
    print("Trend: Negative shift detected")
else:
    print("Trend: No significant negative shift detected")

if safety_flag:
    print("Safety Status: Safety concern detected")
else:
    print("Safety Status: No immediate safety concern detected")

===== MOOD JOURNAL ANALYSIS =====
Journal:

Today was difficult and I felt frustrated throughout the day.
Nothing seemed enjoyable, and I felt very negative.


Current Sentiment: Negative
Current Score: -0.27
Previous Score: 0.2
Trend: Negative shift detected
Safety Status: No immediate safety concern detected


In [18]:
def analyze_mood(journal):

    score = TextBlob(journal).sentiment.polarity

    if score > 0.1:
        sentiment = "Positive"
    elif score < -0.1:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    safety_words = [
        "suicide",
        "kill myself",
        "end my life",
        "self harm",
        "hurt myself"
    ]

    safety_flag = any(
        word in journal.lower()
        for word in safety_words
    )

    print("===== MOOD ANALYSIS =====")
    print("Sentiment:", sentiment)
    print("Score:", round(score, 2))

    if safety_flag:
        print("Safety Status: Safety concern detected.")
        print("Please reach out to a trusted adult or appropriate emergency/crisis support.")
    else:
        print("Safety Status: No immediate safety concern detected.")


# Test
analyze_mood(
    "Today I felt happy and enjoyed my day with my friends."
)

===== MOOD ANALYSIS =====
Sentiment: Positive
Score: 0.65
Safety Status: No immediate safety concern detected.


In [19]:
def analyze_mood_shift(previous_entries, current_entry):

    previous_score = TextBlob(previous_entries).sentiment.polarity
    current_score = TextBlob(current_entry).sentiment.polarity

    print("===== MOOD SHIFT ANALYSIS =====")
    print("Previous Score:", round(previous_score, 2))
    print("Current Score:", round(current_score, 2))

    if current_score < previous_score - 0.2:
        print("Negative shift detected")
    else:
        print("No significant negative shift detected")


previous_entries = """
I had a great day at college.
I enjoyed spending time with my friends.
"""

current_entry = """
Today was difficult.
I felt frustrated and nothing seemed enjoyable.
"""

analyze_mood_shift(previous_entries, current_entry)

===== MOOD SHIFT ANALYSIS =====
Previous Score: 0.65
Current Score: -0.23
Negative shift detected


In [20]:
def complete_mood_analysis(previous_entries, current_entry):

    previous_score = TextBlob(previous_entries).sentiment.polarity
    current_score = TextBlob(current_entry).sentiment.polarity

    if current_score > 0.1:
        sentiment = "Positive"
    elif current_score < -0.1:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    negative_shift = current_score < previous_score - 0.2

    safety_words = [
        "suicide",
        "kill myself",
        "end my life",
        "self harm",
        "hurt myself"
    ]

    safety_flag = any(
        word in current_entry.lower()
        for word in safety_words
    )

    print("===== COMPLETE MOOD ANALYSIS =====")
    print("Current Sentiment:", sentiment)
    print("Current Score:", round(current_score, 2))
    print("Previous Score:", round(previous_score, 2))

    if negative_shift:
        print("Mood Trend: Negative shift detected")
    else:
        print("Mood Trend: No significant negative shift detected")

    if safety_flag:
        print("Safety Status: Safety concern detected")
        print("Please reach out to a trusted adult or appropriate emergency/crisis support.")
    else:
        print("Safety Status: No immediate safety concern detected")


complete_mood_analysis(
    previous_entries,
    current_entry
)

===== COMPLETE MOOD ANALYSIS =====
Current Sentiment: Negative
Current Score: -0.23
Previous Score: 0.65
Mood Trend: Negative shift detected
Safety Status: No immediate safety concern detected


In [21]:
previous_entries = """
Today was good.
I enjoyed my college classes and worked on my project.
"""

current_entry = """
Today was okay.
I attended class and completed some work.
"""

complete_mood_analysis(
    previous_entries,
    current_entry
)

===== COMPLETE MOOD ANALYSIS =====
Current Sentiment: Positive
Current Score: 0.5
Previous Score: 0.6
Mood Trend: No significant negative shift detected
Safety Status: No immediate safety concern detected


In [22]:
from IPython.display import display
import ipywidgets as widgets

journal_box = widgets.Textarea(
    placeholder="Write your journal entry here...",
    description="Journal:",
    layout=widgets.Layout(width="80%", height="120px")
)

analyze_button = widgets.Button(
    description="Analyze Mood"
)

output = widgets.Output()


def analyze_button_clicked(b):
    with output:
        output.clear_output()

        current_entry = journal_box.value

        if current_entry.strip() == "":
            print("Please enter a journal entry.")
            return

        complete_mood_analysis(
            previous_entries,
            current_entry
        )


analyze_button.on_click(analyze_button_clicked)

display(journal_box)
display(analyze_button)
display(output)

Textarea(value='', description='Journal:', layout=Layout(height='120px', width='80%'), placeholder='Write your…

Button(description='Analyze Mood', style=ButtonStyle())

Output()

In [23]:
import ipywidgets as widgets
from IPython.display import display

login_days = widgets.IntText(description="Login Days:")
session_minutes = widgets.IntText(description="Session Min:")
features_used = widgets.IntText(description="Features Used:")
notifications = widgets.IntText(description="Notifications:")

predict_button = widgets.Button(description="Predict Retention")
retention_output = widgets.Output()


def predict_retention(b):
    with retention_output:
        retention_output.clear_output()

        user = pd.DataFrame([{
            "login_days": login_days.value,
            "session_minutes": session_minutes.value,
            "features_used": features_used.value,
            "notifications_opened": notifications.value
        }])

        prediction = model.predict(user)

        if prediction[0] == 1:
            print("Prediction: User is likely to be retained")
        else:
            print("Prediction: User may be at risk of churn")


predict_button.on_click(predict_retention)

display(login_days)
display(session_minutes)
display(features_used)
display(notifications)
display(predict_button)
display(retention_output)

IntText(value=0, description='Login Days:')

IntText(value=0, description='Session Min:')

IntText(value=0, description='Features Used:')

IntText(value=0, description='Notifications:')

Button(description='Predict Retention', style=ButtonStyle())

Output()

In [24]:
import joblib

joblib.dump(model, "retention_model.pkl")

print("Retention model saved successfully!")


Retention model saved successfully!


In [25]:
from google.colab import files

files.download("retention_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>